## Methodologies for Adapting Local Large Language Models for Reliable Transformation of Unstructured Clinical Notes into Structured EHR Data
**Master thesis by Mohammad Omar Sharieff** (target platform: OpenEMR, canonical model: HL7 FHIR R4)

**Notebook 01: data preparation.** Loads the raw MTSamples export, cleans it, draws a seeded random sample, and splits it into a development set (may be inspected freely) and a test set (touched once, at the very end). The notes are *not* gold standard yet; they only become gold standard after manual annotation against the target schema.

In [1]:
import hashlib, json, platform
from pathlib import Path
import pandas as pd

SEED = 42
RAW_PATH = Path("data/raw/mtsamples.csv")      # raw data: never modify
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Specialties kept (note: "Consult - History and Phy." is NOT necessarily outpatient)
SPECIALTIES = ["Consult - History and Phy.", "General Medicine"]
N_DEV, N_TEST = 40, 40                          # adjust to what you can realistically annotate

### 1. Load raw data (with provenance)

In [3]:
raw_sha256 = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
print("Raw file SHA-256:", raw_sha256)

df = pd.read_csv(RAW_PATH, index_col=0)
df.index.name = "source_id"          # keep the original row id as a stable identifier
df = df.reset_index()

print(f"Total notes loaded: {len(df)}")
print("Columns:", df.columns.tolist())

Raw file SHA-256: 83a65fd7ca071930264e27e6fb011bc5303a21294ff734d7a8198cfbb1db7a2a
Total notes loaded: 4999
Columns: ['source_id', 'description', 'medical_specialty', 'sample_name', 'transcription', 'keywords']


### 2. Clean *before* counting or filtering

In [4]:
n0 = len(df)

# strip whitespace first, so value_counts and filters see the same labels
df["medical_specialty"] = df["medical_specialty"].str.strip()
df["transcription"] = df["transcription"].str.strip()

# drop missing / empty transcriptions
df = df.dropna(subset=["transcription"])
df = df[df["transcription"] != ""]
n1 = len(df)

# MTSamples contains duplicate transcriptions; duplicates across dev/test would leak
df = df.drop_duplicates(subset=["transcription"])
n2 = len(df)

print(f"loaded {n0} -> non-empty {n1} -> deduplicated {n2}")
print("\nTop 10 specialties:")
print(df["medical_specialty"].value_counts().head(10))

loaded 4999 -> non-empty 4966 -> deduplicated 2357

Top 10 specialties:
medical_specialty
Surgery                          976
Radiology                        248
General Medicine                 157
Urology                          156
SOAP / Chart / Progress Notes    144
Neurology                         67
Orthopedic                        56
Consult - History and Phy.        55
Pediatrics - Neonatal             52
Psychiatry / Psychology           51
Name: count, dtype: int64


### 3. Filter to encounter-type notes

In [5]:
filtered = df[df["medical_specialty"].isin(SPECIALTIES)].copy()
filtered["note_id"] = filtered["source_id"].map(lambda i: f"mts_{i:04d}")
filtered["n_chars"] = filtered["transcription"].str.len()

print(f"Filtered encounter notes: {len(filtered)}")
print(filtered["medical_specialty"].value_counts())
print("\nNote length (characters):")
print(filtered["n_chars"].describe())

Filtered encounter notes: 212
medical_specialty
General Medicine              157
Consult - History and Phy.     55
Name: count, dtype: int64

Note length (characters):
count      212.000000
mean      3485.415094
std       1669.680393
min         72.000000
25%       2380.000000
50%       3232.500000
75%       4311.250000
max      12223.000000
Name: n_chars, dtype: float64


### 4. Seeded random sample, then dev/test split
The split is decided here, once, before any annotation or prompt work. The test file must not be opened until the final evaluation.

In [6]:
sample = filtered.sample(n=N_DEV + N_TEST, random_state=SEED).reset_index(drop=True)
dev = sample.iloc[:N_DEV].copy()
test = sample.iloc[N_DEV:].copy()

assert set(dev["note_id"]).isdisjoint(test["note_id"])
assert set(dev["transcription"]).isdisjoint(test["transcription"])

print("Dev specialty mix:\n", dev["medical_specialty"].value_counts(), sep="")
print("\nTest specialty mix:\n", test["medical_specialty"].value_counts(), sep="")

Dev specialty mix:
medical_specialty
General Medicine              29
Consult - History and Phy.    11
Name: count, dtype: int64

Test specialty mix:
medical_specialty
General Medicine              30
Consult - History and Phy.    10
Name: count, dtype: int64


### 5. Save (CSV/JSONL for notes, no Excel)
Excel cells are capped at 32,767 characters and silently mangle long text, so notes are stored as CSV/JSONL. Excel/spreadsheet is only worth it later for an annotation sheet.

In [7]:
cols = ["note_id", "source_id", "medical_specialty", "sample_name", "description", "keywords", "transcription"]

filtered[cols].to_csv(OUT_DIR / "all_encounters.csv", index=False)
dev[cols].to_json(OUT_DIR / "dev_notes.jsonl", orient="records", lines=True, force_ascii=False)
test[cols].to_json(OUT_DIR / "test_notes.jsonl", orient="records", lines=True, force_ascii=False)

manifest = {
    "seed": SEED,
    "raw_file": str(RAW_PATH),
    "raw_sha256": raw_sha256,
    "specialties": SPECIALTIES,
    "counts": {"loaded": n0, "non_empty": n1, "deduplicated": n2,
               "filtered": len(filtered), "dev": len(dev), "test": len(test)},
    "dev_ids": dev["note_id"].tolist(),
    "test_ids": test["note_id"].tolist(),
    "python": platform.python_version(),
    "pandas": pd.__version__,
}
(OUT_DIR / "split_manifest.json").write_text(json.dumps(manifest, indent=2))
print("Saved dev/test notes and split_manifest.json to", OUT_DIR)

Saved dev/test notes and split_manifest.json to data\processed
